# CSE475: NinaPro **DB1** EMG Gesture Classification — Task 3: Ablation & Final Model



In [1]:
import os


DATA_PATH = "/kaggle/input/datasets/hoixding/nianpro-db1"  

SEED = 42
FS = 100                      # DB1 EMG sampling rate (Hz)
WINDOW_MS = 200                # analysis window length
STEP_MS = 200                  # step size -> 0% overlap (halves window count vs. 100ms; raise back to 100 for more data if you have time budget)
WINDOW_SIZE = int(FS * WINDOW_MS / 1000)   # = 20 samples
STEP_SIZE = int(FS * STEP_MS / 1000)       # = 10 samples
NUM_CHANNELS = 10              # DB1: 8 equally-spaced forearm electrodes + 2 on main activity muscles
EXERCISES_TO_USE = ["E1"]     
                               
DROP_REST_CLASS = False        
OUTPUT_DIR = "/kaggle/working/task3_ablation"
BASELINE_MODELS_DIR = "/kaggle/working/task2_baselines/models"  # from Notebook 1
FEATURE_CACHE_CSV = "/kaggle/working/db1_windowed_features.csv"  # shared cache across notebooks
os.makedirs(OUTPUT_DIR, exist_ok=True)

import numpy as np, random
np.random.seed(SEED)
random.seed(SEED)

CORR_EDGE_THRESHOLD = 0.3
FAST_EPOCHS = 15               
FAST_PATIENCE = 5
FINAL_EPOCHS = 80
FINAL_PATIENCE = 12
BATCH_SIZE = 256               
N_CV_FOLDS = 5


In [2]:
import torch
try:
    import torch_geometric
except ImportError:
    !pip install -q torch_geometric
    import torch_geometric
print("PyG:", torch_geometric.__version__)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
PyG: 2.8.0.post1
Device: cuda


## 1. Load DB1 + window into TD5 features (reuses cache from Notebooks 1/2 if present)

In [3]:
import glob, re

mat_files = glob.glob(os.path.join(DATA_PATH, "**", "*.mat"), recursive=True)
print(f"Found {len(mat_files)} .mat file(s) under DATA_PATH.")
if len(mat_files) == 0:
    print("Nothing found — listing /kaggle/input to help you fix DATA_PATH:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files[:20]:
            print(os.path.join(root, f))
    raise FileNotFoundError("No .mat files found under DATA_PATH — update DATA_PATH above.")

print("\nSample filenames:")
for f in mat_files[:8]:
    print(" ", os.path.basename(f))

import scipy.io as sio
sample = sio.loadmat(mat_files[0])
sample_keys = [k for k in sample.keys() if not k.startswith("__")]
print("\nKeys inside a sample .mat file:", sample_keys)
for k in sample_keys:
    try:
        print(f"  {k}: shape {sample[k].shape}, dtype {sample[k].dtype}")
    except Exception:
        print(f"  {k}: {type(sample[k])}")


Found 81 .mat file(s) under DATA_PATH.

Sample filenames:
  S26_A1_E2.mat
  S26_A1_E3.mat
  S26_A1_E1.mat
  S20_A1_E3.mat
  S20_A1_E2.mat
  S20_A1_E1.mat
  S18_A1_E1.mat
  S18_A1_E3.mat

Keys inside a sample .mat file: ['subject', 'exercise', 'stimulus', 'emg', 'glove', 'restimulus', 'repetition', 'rerepetition']
  subject: shape (1, 1), dtype uint8
  exercise: shape (1, 1), dtype uint8
  stimulus: shape (144111, 1), dtype uint8
  emg: shape (144111, 10), dtype float64
  glove: shape (144111, 22), dtype float64
  restimulus: shape (144111, 1), dtype uint8
  repetition: shape (144111, 1), dtype uint8
  rerepetition: shape (144111, 1), dtype uint8


In [4]:
import scipy.io as sio
import re as _re

FNAME_RE = _re.compile(r'S(\d+)_A(\d+)_E(\d+)\.mat', _re.IGNORECASE)

def parse_filename(path):
    m = FNAME_RE.search(os.path.basename(path))
    if m:
        return int(m.group(1)), int(m.group(3))  # subject, exercise number (1,2,3 -> E1,E2,E3)
    return None, None

def load_db1_mat(path):
    """Loads one DB1 .mat file, returns dict with emg[T,10], label[T], rep[T], subject(int)."""
    d = sio.loadmat(path)
    emg = d["emg"]
    
    label = d["restimulus"].ravel() if "restimulus" in d else d["stimulus"].ravel()
    rep = d["rerepetition"].ravel() if "rerepetition" in d else d["repetition"].ravel()
    subj_from_file, exercise_num = parse_filename(path)
    subject = int(d["subject"].ravel()[0]) if "subject" in d else subj_from_file
    assert emg.shape[1] >= NUM_CHANNELS, f"Expected >= {NUM_CHANNELS} EMG channels, got {emg.shape[1]} in {path}"
    emg = emg[:, :NUM_CHANNELS]
    return dict(emg=emg, label=label, rep=rep, subject=subject, exercise=exercise_num)


In [5]:
def _mav(w):  return np.mean(np.abs(w), axis=0)
def _rms(w):  return np.sqrt(np.mean(w ** 2, axis=0))
def _wl(w):   return np.sum(np.abs(np.diff(w, axis=0)), axis=0)

def _zc(w, thresh=1e-4):
    sign_change = (w[:-1] * w[1:]) < 0
    big_enough = np.abs(w[:-1] - w[1:]) >= thresh
    return np.sum(sign_change & big_enough, axis=0)

def _ssc(w, thresh=1e-4):
    d1 = w[1:-1] - w[:-2]
    d2 = w[1:-1] - w[2:]
    cond = (d1 * d2) > 0
    big_enough = (np.abs(d1) >= thresh) | (np.abs(d2) >= thresh)
    return np.sum(cond & big_enough, axis=0)

FEATURE_FUNCS = {"MAV": _mav, "RMS": _rms, "WL": _wl, "ZC": _zc, "SSC": _ssc}
FEATURE_NAMES = list(FEATURE_FUNCS.keys())

def extract_windows(rec):
    """Slides a WINDOW_SIZE/STEP_SIZE window over each contiguous
    (repetition, label) segment so windows never straddle a label/repetition
    boundary (which would create ambiguous or leaky training examples)."""
    emg, label, rep, subject, exercise = (rec["emg"], rec["label"], rec["rep"],
                                           rec["subject"], rec["exercise"])
    rows = []
    change_points = np.where((np.diff(label) != 0) | (np.diff(rep) != 0))[0] + 1
    bounds = np.concatenate(([0], change_points, [len(label)]))
    for start, end in zip(bounds[:-1], bounds[1:]):
        seg_len = end - start
        if seg_len < WINDOW_SIZE:
            continue
        seg_label = label[start]
        seg_rep = rep[start]
        if DROP_REST_CLASS and seg_label == 0:
            continue
        for w_start in range(start, end - WINDOW_SIZE + 1, STEP_SIZE):
            w = emg[w_start:w_start + WINDOW_SIZE]
            feat_row = {"subject": subject, "exercise": exercise,
                        "repetition": int(seg_rep), "label": int(seg_label)}
            for fname, ffunc in FEATURE_FUNCS.items():
                vals = ffunc(w)  # shape [NUM_CHANNELS]
                for ch in range(NUM_CHANNELS):
                    feat_row[f"ch{ch+1}_{fname}"] = float(vals[ch])
            rows.append(feat_row)
    return rows


In [6]:
import pandas as pd

if os.path.exists(FEATURE_CACHE_CSV):
    print(f"Loading cached windowed feature table from {FEATURE_CACHE_CSV}")
    df = pd.read_csv(FEATURE_CACHE_CSV)
else:
    selected_files = []
    for f in mat_files:
        _, ex_num = parse_filename(f)
        if ex_num is None:
            continue
        if f"E{ex_num}" in EXERCISES_TO_USE:
            selected_files.append(f)
    print(f"Using {len(selected_files)} file(s) matching EXERCISES_TO_USE={EXERCISES_TO_USE} "
          f"out of {len(mat_files)} total.")
    assert len(selected_files) > 0, "No files matched EXERCISES_TO_USE — check filename pattern / config."

    all_rows = []
    for i, f in enumerate(selected_files):
        rec = load_db1_mat(f)
        rows = extract_windows(rec)
        all_rows.extend(rows)
        if (i + 1) % 10 == 0 or (i + 1) == len(selected_files):
            print(f"  processed {i+1}/{len(selected_files)} files, "
                  f"{len(all_rows)} windows so far...")

    df = pd.DataFrame(all_rows)
    df.to_csv(FEATURE_CACHE_CSV, index=False)
    print(f"Cached windowed feature table to {FEATURE_CACHE_CSV}")

SUBJECT_COL = "subject"
LABEL_COL = "label"
FEATURE_COLS = [c for c in df.columns if c not in ("subject", "exercise", "repetition", "label")]

print("\nWindowed table shape:", df.shape)
print("Subjects:", df[SUBJECT_COL].nunique(), "| Classes:", df[LABEL_COL].nunique(),
      "| Feature columns:", len(FEATURE_COLS))
print("Class distribution:\n", df[LABEL_COL].value_counts().sort_index())
assert df.isna().sum().sum() == 0, "NaNs found in windowed feature table — investigate before proceeding."


Using 27 file(s) matching EXERCISES_TO_USE=['E1'] out of 81 total.
  processed 10/27 files, 49483 windows so far...
  processed 20/27 files, 98857 windows so far...
  processed 27/27 files, 133486 windows so far...
Cached windowed feature table to /kaggle/working/db1_windowed_features.csv

Windowed table shape: (133486, 54)
Subjects: 27 | Classes: 13 | Feature columns: 50
Class distribution:
 label
0     75590
1      5008
2      4950
3      5560
4      4620
5      4634
6      4840
7      4898
8      5077
9      4669
10     4491
11     4309
12     4840
Name: count, dtype: int64


In [7]:
NODE_FEAT_DIM = len(FEATURE_NAMES)
classes = np.sort(df[LABEL_COL].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)

def row_to_node_features(row):
    mat = np.zeros((NUM_CHANNELS, NODE_FEAT_DIM), dtype=np.float32)
    for ch in range(1, NUM_CHANNELS + 1):
        for j, fname in enumerate(FEATURE_NAMES):
            mat[ch - 1, j] = row[f"ch{ch}_{fname}"]
    return mat

print(f"Nodes {NUM_CHANNELS} | NodeFeatDim {NODE_FEAT_DIM} | Classes {NUM_CLASSES}")


Nodes 10 | NodeFeatDim 5 | Classes 13


## 2. Reusable split + graph-construction helpers

Includes both graph variants needed for **Ablation 4**: the correlation/kNN functional graph used as the
default (same as Notebook 2), and a fixed **anatomical adjacency** graph — the 8 equally-spaced forearm
electrodes wired as a ring, with the 2 extra muscle-specific electrodes (channels 9–10, per DB1's
placement on the main flexor/extensor activity muscles) each linked to their two nearest ring
electrodes.

In [8]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

def subject_split(df_, seed, test_frac=0.2, val_frac=0.2):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(df_, groups=df_[SUBJECT_COL]))
    df_trainval, df_test = df_.iloc[trainval_idx], df_.iloc[test_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    train_idx, val_idx = next(gss2.split(df_trainval, groups=df_trainval[SUBJECT_COL]))
    df_train, df_val = df_trainval.iloc[train_idx], df_trainval.iloc[val_idx]
    ts, vs, tes = set(df_train[SUBJECT_COL]), set(df_val[SUBJECT_COL]), set(df_test[SUBJECT_COL])
    assert ts.isdisjoint(vs) and ts.isdisjoint(tes) and vs.isdisjoint(tes), "LEAKAGE in subject_split!"
    return df_train, df_val, df_test

def build_correlation_graph(df_train_, threshold, weighted=True):
    mat = np.zeros((len(df_train_), NUM_CHANNELS), dtype=np.float32)
    for i, (_, row) in enumerate(df_train_.iterrows()):
        mat[i] = row_to_node_features(row).mean(axis=1)
    corr = np.nan_to_num(np.corrcoef(mat, rowvar=False), nan=0.0)
    src, dst, w = [], [], []
    for i in range(NUM_CHANNELS):
        for j in range(NUM_CHANNELS):
            if i != j and abs(corr[i, j]) >= threshold:
                src.append(i); dst.append(j); w.append(abs(corr[i, j]) if weighted else 1.0)
    if len(src) < NUM_CHANNELS:
        k = min(4, NUM_CHANNELS - 1)
        src, dst, w = [], [], []
        for i in range(NUM_CHANNELS):
            order = np.argsort(-np.abs(corr[i])); order = order[order != i][:k]
            for j in order:
                src.append(i); dst.append(int(j)); w.append(abs(corr[i, j]) if weighted else 1.0)
    return torch.tensor([src, dst], dtype=torch.long), torch.tensor(w, dtype=torch.float32)

def build_anatomical_adjacency_graph():
    # channels 0-7 (ch1..ch8): ring of 8 equally-spaced forearm electrodes
    # channels 8-9 (ch9, ch10): extra electrodes on main flexor/extensor muscles ->
    #                           linked to their two nearest ring electrodes (assumption, documented)
    src, dst, w = [], [], []
    ring = list(range(8))
    for idx, i in enumerate(ring):
        for j in (ring[(idx - 1) % 8], ring[(idx + 1) % 8]):
            src.append(i); dst.append(j); w.append(1.0)
    for extra in (8, 9):
        for j in (0, 4):  # connect each extra electrode to two roughly-opposite ring nodes
            src.append(extra); dst.append(j); w.append(1.0)
            src.append(j); dst.append(extra); w.append(1.0)
    return torch.tensor([src, dst], dtype=torch.long), torch.tensor(w, dtype=torch.float32)

def df_to_graphs(df_subset, edge_index, edge_weight):
    from torch_geometric.data import Data
    graphs = []
    for _, row in df_subset.iterrows():
        yv = row[LABEL_COL]
        if yv not in class_to_idx:
            continue
        x = torch.tensor(row_to_node_features(row), dtype=torch.float32)
        y = torch.tensor([class_to_idx[yv]], dtype=torch.long)
        graphs.append(Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y))
    return graphs


## 3. Configurable GNN model (architecture / layers / hidden dim / dropout / pooling)

In [9]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATv2Conv, SAGEConv, global_mean_pool, global_max_pool

def make_conv(arch, in_dim, out_dim, heads=4, concat=True):
    if arch == "GCN":  return GCNConv(in_dim, out_dim)
    if arch == "GAT":  return GATv2Conv(in_dim, out_dim // heads if concat else out_dim,
                                         heads=heads, concat=concat, edge_dim=1)
    if arch == "SAGE": return SAGEConv(in_dim, out_dim)
    raise ValueError(arch)

class ConfigurableGNN(nn.Module):
    def __init__(self, in_dim, num_classes, arch="GAT", hidden_dim=64, num_layers=2,
                 dropout=0.3, pooling="mean_max"):
        super().__init__()
        self.arch, self.pooling, self.dropout = arch, pooling, dropout
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        dims = [in_dim] + [hidden_dim] * num_layers
        for i in range(num_layers):
            concat = (arch == "GAT") and (i < num_layers - 1)
            self.convs.append(make_conv(arch, dims[i], hidden_dim, concat=concat))
            self.norms.append(nn.BatchNorm1d(hidden_dim))
        pool_mult = 2 if pooling == "mean_max" else 1
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * pool_mult, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes))

    def forward(self, x, edge_index, edge_attr, batch):
        for conv, norm in zip(self.convs, self.norms):
            x = conv(x, edge_index, edge_attr=edge_attr.unsqueeze(-1)) if self.arch == "GAT" \
                else conv(x, edge_index)
            x = F.relu(norm(x))
            x = F.dropout(x, p=self.dropout, training=self.training)
        pooled = (global_mean_pool(x, batch) if self.pooling == "mean" else
                  global_max_pool(x, batch) if self.pooling == "max" else
                  torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1))
        return self.classifier(pooled)


## 4. Unified train/eval function, parameterized by a single `config` dict
Node features (MAV/RMS/WL/ZC/SSC) live on very different numeric scales, so they are standardized (fit on the train split passed into this call, applied to val/test — leakage-safe and correctly re-fit for every ablation run and every CV fold) before graph construction.

In [10]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from torch_geometric.loader import DataLoader as PyGDataLoader
import time

DEFAULT_CONFIG = dict(arch="GAT", hidden_dim=64, num_layers=2, dropout=0.3, pooling="mean_max",
                       edge_weighted=True, graph_type="correlation", optimizer="AdamW",
                       use_scheduler=True, weighted_loss=True, epochs=FAST_EPOCHS, patience=FAST_PATIENCE)

def run_experiment(config, df_train, df_val, df_test, verbose=False):
    cfg = {**DEFAULT_CONFIG, **config}

    # Normalize node features -- fit on THIS call's train split only.
    node_scaler = StandardScaler().fit(df_train[FEATURE_COLS].values)
    df_train = df_train.copy(); df_val = df_val.copy(); df_test = df_test.copy()
    df_train[FEATURE_COLS] = node_scaler.transform(df_train[FEATURE_COLS].values)
    df_val[FEATURE_COLS]   = node_scaler.transform(df_val[FEATURE_COLS].values)
    df_test[FEATURE_COLS]  = node_scaler.transform(df_test[FEATURE_COLS].values)

    if cfg["graph_type"] == "adjacency":
        edge_index, edge_weight = build_anatomical_adjacency_graph()
    else:
        edge_index, edge_weight = build_correlation_graph(df_train, CORR_EDGE_THRESHOLD, weighted=cfg["edge_weighted"])
    if not cfg["edge_weighted"]:
        edge_weight = torch.ones_like(edge_weight)

    train_loader = PyGDataLoader(df_to_graphs(df_train, edge_index, edge_weight), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = PyGDataLoader(df_to_graphs(df_val, edge_index, edge_weight), batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = PyGDataLoader(df_to_graphs(df_test, edge_index, edge_weight), batch_size=BATCH_SIZE, shuffle=False)

    model = ConfigurableGNN(NODE_FEAT_DIM, NUM_CLASSES, arch=cfg["arch"], hidden_dim=cfg["hidden_dim"],
                             num_layers=cfg["num_layers"], dropout=cfg["dropout"], pooling=cfg["pooling"]).to(DEVICE)

    y_train_labels = np.array([class_to_idx[v] for v in df_train[LABEL_COL].values if v in class_to_idx])
    weight = None
    if cfg["weighted_loss"]:
        cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train_labels)
        weight = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight)

    opt_cls = torch.optim.AdamW if cfg["optimizer"] == "AdamW" else torch.optim.Adam
    optimizer = opt_cls(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)
                 if cfg["use_scheduler"] else None)

    def epoch_pass(loader, train_mode):
        model.train() if train_mode else model.eval()
        preds, trues, tot_loss = [], [], 0.0
        with torch.set_grad_enabled(train_mode):
            for batch in loader:
                batch = batch.to(DEVICE)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = criterion(out, batch.y)
                if train_mode:
                    optimizer.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                    optimizer.step()
                tot_loss += loss.item() * batch.num_graphs
                preds.append(out.argmax(1).detach().cpu().numpy())
                trues.append(batch.y.detach().cpu().numpy())
        preds, trues = np.concatenate(preds), np.concatenate(trues)
        return tot_loss / len(loader.dataset), f1_score(trues, preds, average="macro", zero_division=0)

    best_val_f1, best_state, no_improve = -1, None, 0
    t0 = time.time()
    for ep in range(1, cfg["epochs"] + 1):
        epoch_pass(train_loader, True)
        _, val_f1 = epoch_pass(val_loader, False)
        if scheduler is not None:
            scheduler.step(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= cfg["patience"]:
            break
    train_time = time.time() - t0
    model.load_state_dict(best_state)

    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
            all_preds.append(out.argmax(1).cpu().numpy()); all_true.append(batch.y.cpu().numpy())
    all_preds, all_true = np.concatenate(all_preds), np.concatenate(all_true)
    acc = accuracy_score(all_true, all_preds)
    prec, rec, f1m, _ = precision_recall_fscore_support(all_true, all_preds, average="macro", zero_division=0)
    _, _, f1w, _ = precision_recall_fscore_support(all_true, all_preds, average="weighted", zero_division=0)
    if verbose:
        print(f"  -> val_best_f1={best_val_f1:.4f} test_acc={acc:.4f} test_macroF1={f1m:.4f} ({train_time:.0f}s)")
    return dict(Accuracy=acc, MacroF1=f1m, WeightedF1=f1w, Precision=prec, Recall=rec,
                train_time_s=train_time, val_best_f1=best_val_f1)


## 5. Ablation sweep (fast mode — one held-out split, reused across all experiments)

In [11]:
import json
df_train, df_val, df_test = subject_split(df, seed=SEED)
print("Ablation split sizes:", len(df_train), len(df_val), len(df_test))

experiments = [("Baseline GNN", dict())]
for n in (1, 2, 3):                      experiments.append((f"A1_layers_{n}", dict(num_layers=n)))
for h in (32, 64, 128):                  experiments.append((f"A2_hidden_{h}", dict(hidden_dim=h)))
for d in (0.0, 0.2, 0.5):                experiments.append((f"A3_dropout_{d}", dict(dropout=d)))
experiments.append(("A4_graph_adjacency", dict(graph_type="adjacency")))
experiments.append(("A4_graph_correlation", dict(graph_type="correlation")))
for arch in ("GCN", "GAT", "SAGE"):      experiments.append((f"A5_arch_{arch}", dict(arch=arch)))
experiments.append(("A6_edge_unweighted", dict(edge_weighted=False)))
experiments.append(("A6_edge_weighted", dict(edge_weighted=True)))
for p in ("mean", "max", "mean_max"):    experiments.append((f"A7_pool_{p}", dict(pooling=p)))
experiments.append(("A8_optim_Adam", dict(optimizer="Adam")))
experiments.append(("A8_optim_AdamW", dict(optimizer="AdamW")))
experiments.append(("A8_no_scheduler", dict(use_scheduler=False)))
experiments.append(("A8_unweighted_loss", dict(weighted_loss=False)))

print(f"Running {len(experiments)} experiments in fast mode ({FAST_EPOCHS} max epochs each)...")
ablation_rows = []
for name, cfg in experiments:
    print(f"[{name}] config={cfg}")
    metrics = run_experiment(cfg, df_train, df_val, df_test, verbose=True)
    ablation_rows.append({"Experiment": name, "Configuration": json.dumps(cfg), **metrics})

ablation_df = pd.DataFrame(ablation_rows).sort_values("MacroF1", ascending=False).reset_index(drop=True)
ablation_df.to_csv(os.path.join(OUTPUT_DIR, "ablation_results.csv"), index=False)
print(ablation_df[["Experiment", "Configuration", "Accuracy", "MacroF1", "WeightedF1"]].to_string(index=False))


Ablation split sizes: 79162 24704 29620
Running 24 experiments in fast mode (15 max epochs each)...
[Baseline GNN] config={}


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  -> val_best_f1=0.1490 test_acc=0.4831 test_macroF1=0.1269 (47s)
[A1_layers_1] config={'num_layers': 1}


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  -> val_best_f1=0.1541 test_acc=0.4893 test_macroF1=0.1396 (53s)
[A1_layers_2] config={'num_layers': 2}


...

  -> val_best_f1=0.1417 test_acc=0.4753 test_macroF1=0.1226 (77s)
[A8_no_scheduler] config={'use_scheduler': False}


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  -> val_best_f1=0.1486 test_acc=0.4928 test_macroF1=0.1201 (54s)
[A8_unweighted_loss] config={'weighted_loss': False}


/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` 

  -> val_best_f1=0.1373 test_acc=0.5499 test_macroF1=0.1090 (115s)
          Experiment                 Configuration  Accuracy  MacroF1  WeightedF1
         A1_layers_1             {"num_layers": 1}  0.489332 0.139587    0.479333
        A7_pool_mean           {"pooling": "mean"}  0.499696 0.134091    0.480604
        A5_arch_SAGE              {"arch": "SAGE"}  0.488724 0.133882    0.476299
         A1_layers_3             {"num_layers": 3}  0.501823 0.133329    0.475940
    A7_pool_mean_max       {"pooling": "mean_max"}  0.477785 0.132376    0.471168
      A3_dropout_0.0              {"dropout": 0.0}  0.460972 0.131161    0.463826
      A3_dropout_0.2              {"dropout": 0.2}  0.491864 0.130833    0.475492
    A6_edge_weighted       {"edge_weighted": true}  0.496624 0.130308    0.479704
        A2_hidden_64            {"hidden_dim": 64}  0.481026 0.128804    0.472215
      A3_dropout_0.5              {"dropout": 0.5}  0.487441 0.128239    0.473884
         A7_pool_max           

## 6. Pick the best configuration -> assemble the Final GNN config

In [12]:
baseline_f1 = ablation_df.loc[ablation_df["Experiment"] == "Baseline GNN", "MacroF1"].values[0]
final_config = dict(DEFAULT_CONFIG)
axis_prefixes = ["A1_layers", "A2_hidden", "A3_dropout", "A4_graph", "A5_arch", "A6_edge", "A7_pool", "A8_"]
for prefix in axis_prefixes:
    axis_rows = ablation_df[ablation_df["Experiment"].str.startswith(prefix)]
    if len(axis_rows) == 0:
        continue
    best_row = axis_rows.sort_values("MacroF1", ascending=False).iloc[0]
    if best_row["MacroF1"] >= baseline_f1:
        final_config.update(json.loads(best_row["Configuration"]))
final_config["epochs"] = FINAL_EPOCHS
final_config["patience"] = FINAL_PATIENCE

print("Baseline Macro-F1:", f"{baseline_f1:.4f}")
print("Selected FINAL config:", json.dumps(final_config, indent=2))
with open(os.path.join(OUTPUT_DIR, "final_config.json"), "w") as f:
    json.dump(final_config, f, indent=2)


Baseline Macro-F1: 0.1269
Selected FINAL config: {
  "arch": "SAGE",
  "hidden_dim": 64,
  "num_layers": 1,
  "dropout": 0.0,
  "pooling": "mean",
  "edge_weighted": true,
  "graph_type": "adjacency",
  "optimizer": "AdamW",
  "use_scheduler": true,
  "weighted_loss": true,
  "epochs": 80,
  "patience": 12
}


## 7. 5-fold subject-wise CV for the Final GNN (paired with the best baseline)

DB1 has 27 subjects, comfortably enough for a valid 5-fold subject-wise split — but the check below is
kept in place and will automatically reduce `N_CV_FOLDS` (with a clear warning) if run on a smaller
subject subset.

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

n_subjects = df[SUBJECT_COL].nunique()
effective_folds = min(N_CV_FOLDS, n_subjects)
if effective_folds < N_CV_FOLDS:
    print(f"WARNING: only {n_subjects} subjects available — reducing to {effective_folds}-fold CV.")

gkf = GroupKFold(n_splits=effective_folds)
fold_metrics_gnn, fold_metrics_baseline = [], []

for fold_i, (trainval_idx, test_idx) in enumerate(gkf.split(df, groups=df[SUBJECT_COL]), start=1):
    df_trainval_f, df_test_f = df.iloc[trainval_idx], df.iloc[test_idx]
    inner_subjects = df_trainval_f[SUBJECT_COL].unique()
    rng = np.random.RandomState(SEED + fold_i)
    val_subjects_f = set(rng.choice(inner_subjects, size=max(1, len(inner_subjects) // 5), replace=False))
    df_val_f = df_trainval_f[df_trainval_f[SUBJECT_COL].isin(val_subjects_f)]
    df_train_f = df_trainval_f[~df_trainval_f[SUBJECT_COL].isin(val_subjects_f)]

    assert set(df_train_f[SUBJECT_COL]).isdisjoint(set(df_test_f[SUBJECT_COL])), "Fold leakage!"
    assert set(df_val_f[SUBJECT_COL]).isdisjoint(set(df_test_f[SUBJECT_COL])), "Fold leakage!"
    print(f"\n--- Fold {fold_i}/{effective_folds} (train={len(df_train_f)}, val={len(df_val_f)}, test={len(df_test_f)}) ---")

    gnn_res = run_experiment(final_config, df_train_f, df_val_f, df_test_f, verbose=True)
    fold_metrics_gnn.append(gnn_res)

    scaler_f = StandardScaler().fit(df_train_f[FEATURE_COLS].values)
    Xtr = scaler_f.transform(df_train_f[FEATURE_COLS].values)
    Xte = scaler_f.transform(df_test_f[FEATURE_COLS].values)
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED)
    rf.fit(Xtr, df_train_f[LABEL_COL].values)
    pred_rf = rf.predict(Xte)
    _, _, f1m_rf, _ = precision_recall_fscore_support(df_test_f[LABEL_COL].values, pred_rf, average="macro", zero_division=0)
    acc_rf = accuracy_score(df_test_f[LABEL_COL].values, pred_rf)
    fold_metrics_baseline.append({"Accuracy": acc_rf, "MacroF1": f1m_rf})

cv_gnn_df = pd.DataFrame(fold_metrics_gnn)
cv_base_df = pd.DataFrame(fold_metrics_baseline)

def mean_std(s): return f"{s.mean():.4f} +/- {s.std():.4f}"

print("\n=== Final GNN — 5-fold subject-wise CV ===")
for col in ["Accuracy", "MacroF1", "WeightedF1", "Precision", "Recall"]:
    print(f"{col}: {mean_std(cv_gnn_df[col])}")
print("\n=== Best-baseline-family (RandomForest) — same folds ===")
for col in ["Accuracy", "MacroF1"]:
    print(f"{col}: {mean_std(cv_base_df[col])}")

cv_gnn_df.to_csv(os.path.join(OUTPUT_DIR, "cv_gnn_folds.csv"), index=False)
cv_base_df.to_csv(os.path.join(OUTPUT_DIR, "cv_baseline_folds.csv"), index=False)



--- Fold 1/5 (train=89013, val=19740, test=24733) ---
  -> val_best_f1=0.1334 test_acc=0.4186 test_macroF1=0.1562 (85s)

--- Fold 2/5 (train=88975, val=19780, test=24731) ---
  -> val_best_f1=0.1916 test_acc=0.4446 test_macroF1=0.1355 (144s)

--- Fold 3/5 (train=88910, val=19844, test=24732) ---
  -> val_best_f1=0.1954 test_acc=0.4131 test_macroF1=0.1470 (86s)

--- Fold 4/5 (train=84056, val=19785, test=29645) ---
  -> val_best_f1=0.1417 test_acc=0.3223 test_macroF1=0.1483 (91s)

--- Fold 5/5 (train=84132, val=19709, test=29645) ---
  -> val_best_f1=0.1471 test_acc=0.4945 test_macroF1=0.1478 (92s)

=== Final GNN — 5-fold subject-wise CV ===
Accuracy: 0.4186 +/- 0.0627
MacroF1: 0.1470 +/- 0.0074
WeightedF1: 0.4582 +/- 0.0435
Precision: 0.1633 +/- 0.0079
Recall: 0.1634 +/- 0.0123

=== Best-baseline-family (RandomForest) — same folds ===
Accuracy: 0.5978 +/- 0.0364
MacroF1: 0.2238 +/- 0.0369


## 8. Wilcoxon signed-rank test: Final GNN vs. best baseline (paired per fold)

In [14]:
from scipy.stats import wilcoxon
alpha = 0.05
print("Per-fold Macro-F1 (GNN vs baseline):")
for i, (g, b) in enumerate(zip(cv_gnn_df["MacroF1"], cv_base_df["MacroF1"]), start=1):
    print(f"  Fold {i}: GNN={g:.4f}  Baseline={b:.4f}  diff={g-b:+.4f}")

diffs = cv_gnn_df["MacroF1"].values - cv_base_df["MacroF1"].values
if len(diffs) < 6 or np.all(diffs == 0):
    print(f"\nOnly {len(diffs)} paired folds — interpret the p-value cautiously.")
try:
    stat, p_value = wilcoxon(cv_gnn_df["MacroF1"].values, cv_base_df["MacroF1"].values)
    significant = p_value < alpha
    print(f"\nWilcoxon signed-rank test: statistic={stat:.4f}, p-value={p_value:.4f}, alpha={alpha}")
    print(f"Statistically significant difference: {significant}")
except ValueError as e:
    print(f"\nWilcoxon test could not be computed: {e}")
    stat, p_value, significant = np.nan, np.nan, False


Per-fold Macro-F1 (GNN vs baseline):
  Fold 1: GNN=0.1562  Baseline=0.2229  diff=-0.0667
  Fold 2: GNN=0.1355  Baseline=0.1622  diff=-0.0267
  Fold 3: GNN=0.1470  Baseline=0.2297  diff=-0.0827
  Fold 4: GNN=0.1483  Baseline=0.2504  diff=-0.1021
  Fold 5: GNN=0.1478  Baseline=0.2540  diff=-0.1062

Only 5 paired folds — interpret the p-value cautiously.

Wilcoxon signed-rank test: statistic=0.0000, p-value=0.0625, alpha=0.05
Statistically significant difference: False


## 9. Final comparison table (Best Baseline vs. Initial GNN vs. Final GNN)

In [15]:
INITIAL_GNN_METRICS_CSV = "/kaggle/working/task2_gnn/gnn_metrics.csv"
BASELINE_RESULTS_CSV = "/kaggle/working/task2_baselines/baseline_comparison.csv"


rows = []
if os.path.exists(BASELINE_RESULTS_CSV):
    bdf = pd.read_csv(BASELINE_RESULTS_CSV).sort_values("Macro-F1", ascending=False).iloc[0]
    rows.append({"Model": f"Best Baseline ({bdf['Model']})", "Accuracy": bdf["Accuracy"],
                 "Macro-F1": bdf["Macro-F1"], "Weighted-F1": bdf["Weighted-F1"],
                 "Precision": bdf["Precision(macro)"], "Recall": bdf["Recall(macro)"]})
else:
    rows.append({"Model": "Best Baseline (RandomForest, fold estimate)",
                 "Accuracy": cv_base_df["Accuracy"].mean(), "Macro-F1": cv_base_df["MacroF1"].mean(),
                 "Weighted-F1": np.nan, "Precision": np.nan, "Recall": np.nan})

if os.path.exists(INITIAL_GNN_METRICS_CSV):
    idf = pd.read_csv(INITIAL_GNN_METRICS_CSV).iloc[0]
    rows.append({"Model": "Initial Proposed GNN", "Accuracy": idf["Accuracy"], "Macro-F1": idf["Macro-F1"],
                 "Weighted-F1": idf["Weighted-F1"], "Precision": idf["Precision(macro)"], "Recall": idf["Recall(macro)"]})
else:
    print(f"Initial GNN metrics not found at {INITIAL_GNN_METRICS_CSV} — run Notebook 2 first for a complete table.")

rows.append({"Model": "Final Improved GNN (5-fold CV mean)", "Accuracy": cv_gnn_df["Accuracy"].mean(),
             "Macro-F1": cv_gnn_df["MacroF1"].mean(), "Weighted-F1": cv_gnn_df["WeightedF1"].mean(),
             "Precision": cv_gnn_df["Precision"].mean(), "Recall": cv_gnn_df["Recall"].mean()})

final_table = pd.DataFrame(rows)
print(final_table.to_string(index=False))
final_table.to_csv(os.path.join(OUTPUT_DIR, "final_comparison_table.csv"), index=False)
print(f"\nWilcoxon vs. best baseline: statistic={stat}, p-value={p_value}, significant@0.05: {significant}")
print(f"\nAll Task 3 artifacts saved under: {OUTPUT_DIR}")
print("NOTE: add a 'related work' row manually only for papers using a genuinely matching protocol "
      "(same exercise subset, same split style, same metric).")


Initial GNN metrics not found at /kaggle/working/task2_gnn/gnn_metrics.csv — run Notebook 2 first for a complete table.
                                      Model  Accuracy  Macro-F1  Weighted-F1  Precision   Recall
Best Baseline (RandomForest, fold estimate)  0.597765  0.223846          NaN        NaN      NaN
        Final Improved GNN (5-fold CV mean)  0.418608  0.146968     0.458179   0.163312 0.163358

Wilcoxon vs. best baseline: statistic=0.0, p-value=0.0625, significant@0.05: False

All Task 3 artifacts saved under: /kaggle/working/task3_ablation
NOTE: add a 'related work' row manually only for papers using a genuinely matching protocol (same exercise subset, same split style, same metric).
